# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GourabGorai/FlyRankInternship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

Here we construct the clean, un-leaked feature matrix:
- Log-transform heavy-tailed metrics (`log_impressions_90d`, `log_clicks_90d`).
- Engineered update-to-age interaction ratio (`update_ratio = days_since_last_update / (content_age_days + 1)`).
- Explicit missingness and unranked indicator flags (`has_missing_wc`, `is_unranked`).
- Categorical dummy encoding for `content_type`.

In [1]:
import os, sys, pandas as pd, numpy as np
csv_path = 'data/raw/content_refresh_anonymized.csv'
if not os.path.exists(csv_path):
    csv_path = '../../data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(csv_path)
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)

features_num = ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update', 'word_count', 'engagement_rate']
X = df[features_num].copy()
X['log_impressions'] = np.log1p(np.maximum(0, X['impressions_90d']))
X['log_clicks'] = np.log1p(np.maximum(0, X['clicks_90d']))
X['update_ratio'] = X['days_since_last_update'] / (X['content_age_days'] + 1)
X['has_missing_wc'] = X['word_count'].isna().astype(int)
X['is_unranked'] = (X['avg_position'] == 0).astype(int)
X = X.fillna(0)
print(f'Feature vector shape: {X.shape}')


Feature vector shape: (30000, 13)


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Business Meaning | Missing Handling | Available Pre-Decision? |
|---|---|---|---|
| `impressions_90d` / `log_impressions` | Historical organic exposure | Filled with 0 | Yes (pre-decision) |
| `clicks_90d` / `log_clicks` | Historical organic traffic volume | Filled with 0 | Yes (pre-decision) |
| `ctr` | Realized click-through rate percentage | Filled with 0 | Yes (pre-decision) |
| `avg_position` | Average Google ranking position | Imputed 0 + flag | Yes (pre-decision) |
| `content_age_days` | Days since URL initial publication | Filled with 0 | Yes (pre-decision) |
| `days_since_last_update` | Days elapsed since last editorial revision | Filled with 0 | Yes (pre-decision) |
| `update_ratio` | Proportion of article lifespan spent un-updated | Computed ratio | Yes (pre-decision) |
| `word_count` / `has_missing_wc` | Content depth / missing flag | Median / 0 + flag | Yes (pre-decision) |
| `engagement_rate` | GA4 user session engagement | Filled with 0 | Yes (pre-decision) |

In [2]:
summary = pd.DataFrame({
    'dtype': X.dtypes,
    'null_count': X.isnull().sum(),
    'mean': X.mean().round(2),
    'median': X.median().round(2)
})
print(summary)


                          dtype  null_count     mean   median
impressions_90d           int64           0  5200.37   731.00
clicks_90d                int64           0    16.10     1.00
ctr                     float64           0     0.51     0.07
avg_position            float64           0    16.34    10.80
content_age_days          int64           0   256.17   236.00
days_since_last_update    int64           0    46.10    20.00
word_count              float64           0  2310.21  2605.00
engagement_rate         float64           0     2.53     0.00
log_impressions         float64           0     6.19     6.60
log_clicks              float64           0     1.21     0.69
update_ratio            float64           0     0.21     0.15
has_missing_wc            int64           0     0.26     0.00
is_unranked               int64           0     0.04     0.00


## 3. The leakage hunt

**Attacking the model:** We deliberately train a Decision Tree including the leaky column `trend_pct` vs our clean feature set. Because `is_declining_label` is computed directly from `trend_pct`, the leaky model learns a single trivial split (`trend_pct <= -0.05`), achieving artificial 100% precision. This proves our leakage detection harness works, and explains why `trend_pct` must be strictly excluded.

In [3]:
from sklearn.tree import DecisionTreeClassifier, export_text
y = df['is_declining_label'].values

# 1. Leaky model
X_leaky = X.copy()
X_leaky['trend_pct'] = df['trend_pct'].fillna(0)
dt_leaky = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_leaky, y)
p50_leaky = (y[np.argsort(-dt_leaky.predict_proba(X_leaky)[:, 1])[:50]]).mean()
print(f'Leaky Model Precision@50: {p50_leaky:.3f} (Suspicious perfection: target is leaked!)')
print(export_text(dt_leaky, feature_names=list(X_leaky.columns)))

# 2. Clean model
dt_clean = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X, y)
p50_clean = (y[np.argsort(-dt_clean.predict_proba(X)[:, 1])[:50]]).mean()
print(f'Clean Model Precision@50: {p50_clean:.3f} (Honest, non-leaked signal)')


Leaky Model Precision@50: 1.000 (Suspicious perfection: target is leaked!)
|--- trend_pct <= -20.05
|   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0

Clean Model Precision@50: 0.740 (Honest, non-leaked signal)


## 4. What I excluded and why

1. `trend_pct`: Direct derivation source of `trend_direction` and the target label. Excluded to prevent 100% label leakage.
2. `trend_direction`: The string categorical from which `is_declining_label` is formed.
3. `client_id` and `content_id`: Pseudonymized identifiers excluded as features to prevent the model from memorizing specific client domains.
4. Internal decision flags: Any heuristic flag created by legacy SEO software is excluded to avoid learning existing heuristics.

In [4]:
for col in ['trend_pct', 'trend_direction', 'client_id', 'content_id']:
    assert col not in X.columns, f'Leakage alert: {col} is present in features!'
print('Leakage Audit Passed: All prohibited features are strictly excluded from the feature matrix.')


Leakage Audit Passed: All prohibited features are strictly excluded from the feature matrix.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.